# Spectral and scale-aware losses

A pointwise loss compares forecasts and targets one grid point at a time. Scale-aware losses first decompose each field into components that describe its structure at particular scales, then score those components. This lets a loss check whether a forecast reproduces variability at each scale: for example, whether an ensemble has lost its fine-scale detail through smoothing.

This notebook introduces the decompositions AutoCast provides and shows how to use standalone coefficient losses. It simulates a small conditioned Navier–Stokes ensemble with AutoSim, so it needs no trained model or downloaded dataset.

1. Tensor layout and the decomposition contract
2. A simulated ensemble
3. Fourier coefficients: magnitude and phase
4. Radial Fourier bands
5. Discrete wavelets (DWT): localised scale components
6. Dual-tree complex wavelets (DTCWT): adding direction
7. Physical-space scale bands
8. Effect of smoothing on the decomposition

## 1. Tensor layout and the decomposition contract

Targets have channel-last layout

$$
(B, T, *S, C),
$$

and ensemble predictions add a final member axis:

$$
(B, T, *S, C, M).
$$

Here space is two-dimensional, so the layouts are `(B, T, H, W, C)` and `(B, T, H, W, C, M)`.

A `FieldDecomposition` maps a field to named components, keeping the batch, time, channel and optional member axes. `TransformedLoss` decomposes predictions and targets alike, scores each matching pair of components, and aggregates the results. Every loss in this notebook follows that path.

In [ ]:
import matplotlib.pyplot as plt
import torch
from autosim.experimental.simulations import ConditionedNavierStokes2D
from einops import rearrange
from ptwt import fswavedec2, fswaverec2
from pytorch_wavelets import DTCWTForward, DTCWTInverse

from autocast.losses import (
    CoefficientCRPSLoss,
    CoefficientLpLoss,
    ComponentBranch,
    ComponentObjective,
    CRPSLoss,
    DTCWTDecomposition,
    FourierDecomposition,
    FourierLowPass,
    GaussianSmoothing,
    IdentityDecomposition,
    RadialFourierDecomposition,
    ScaleDecomposition,
    TransformedLoss,
    WaveletDecomposition,
)

torch.manual_seed(7)
plt.rcParams["figure.figsize"] = (10, 3)

## 2. A simulated ensemble

`ConditionedNavierStokes2D` simulates buoyancy-driven smoke in a two-dimensional box. Its state has three channels: smoke concentration and the two velocity components `(smoke, u, v)`. Neumann boundaries keep solid walls, so the Fourier examples below analyse the field's periodic extension and can pick up some energy from the seam at the boundary.

Every run starts from the same random initial condition. The target uses reference values for buoyancy, viscosity and smoke diffusivity, and each ensemble member perturbs those parameters. The members are therefore forecasts of the same initial state under uncertain dynamics. The retained snapshots are at times `t = 13, 14, 15`.

In [ ]:
def simulate_cns(
    buoyancy: float, viscosity: float, smoke_diffusivity: float
) -> torch.Tensor:
    """Simulate one forecast from a shared initial condition."""
    simulator = ConditionedNavierStokes2D(
        parameters_range={
            "buoyancy_y": (buoyancy, buoyancy),
            "smoke_diffusivity": (smoke_diffusivity, smoke_diffusivity),
        },
        return_timeseries=True,
        log_level="warning",
        n=32,
        T=15.0,
        dt=0.26,
        snapshot_dt=1.0,
        nu=viscosity,
        bc_mode="neumann",
        skip_nt=12,
        random_seed=17,
    )
    return simulator.forward_samples_spatiotemporal(
        n=1, random_seed=7, ensure_exact_n=True
    )["data"]


target = simulate_cns(buoyancy=0.35, viscosity=0.030, smoke_diffusivity=0.0005)
member_specs = [
    (0.20, 0.020, 0.00010),
    (0.50, 0.045, 0.00100),
    (0.28, 0.040, 0.00030),
    (0.44, 0.022, 0.00085),
]
members = [simulate_cns(*parameters) for parameters in member_specs]
preds = torch.stack(members, dim=-1).detach()

print("target:", tuple(target.shape))
print("predictions:", tuple(preds.shape))

In [ ]:
smoke_frame = target[0, -1, :, :, 0]
field_limits = (smoke_frame.min(), smoke_frame.max())

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
images = [
    (smoke_frame, "Target smoke"),
    (preds[0, -1, :, :, 0, 0].detach(), "One member"),
    (preds[0, -1, :, :, 0].mean(-1).detach(), "Ensemble mean"),
]
for axis, (image, title) in zip(axes, images, strict=True):
    axis.imshow(image, origin="lower", cmap="viridis")
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()

`IdentityDecomposition` returns the field unchanged, so it recovers an ordinary grid-space loss. It is the simplest instance of the decomposition path.

In [ ]:
identity_crps = TransformedLoss(IdentityDecomposition(), CRPSLoss())
torch.testing.assert_close(identity_crps(preds, target), CRPSLoss()(preds, target))
print("The identity path equals ordinary CRPS.")

## 3. Fourier coefficients: magnitude and phase

`FourierDecomposition` takes an FFT over the spatial axes and returns one component, `fourier`. With `representation="complex"` it keeps the coefficients; `"magnitude"` and `"power"` keep their modulus or squared modulus.

Write a coefficient as

$$
x_k = A_k e^{i\varphi_k}.
$$

The magnitude $A_k$ says how much variation the field has at frequency $k$, and the phase $\varphi_k$ says where those oscillations sit. Translating a periodic field changes the phases but not the magnitudes, which makes a translation a useful check.

In [ ]:
translated = torch.roll(target, shifts=(3, -2), dims=(2, 3))
complex_translation_error = TransformedLoss(
    FourierDecomposition("complex"), CoefficientLpLoss(p=1)
)(translated, target)
magnitude_translation_error = TransformedLoss(
    FourierDecomposition("magnitude"), CoefficientLpLoss(p=1)
)(translated, target)

print("Complex-coefficient error:", complex_translation_error.item())
print("Magnitude-only error:     ", magnitude_translation_error.item())

In [ ]:
fourier_components = FourierDecomposition("complex")(target, n_spatial_dims=2)
coefficients = torch.fft.fftshift(fourier_components["fourier"][0, -1, :, :, 0])

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(torch.log1p(coefficients.abs()), origin="lower")
axes[0].set_title("log(1 + magnitude)")
axes[1].imshow(torch.angle(coefficients), origin="lower", cmap="twilight")
axes[1].set_title("phase")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

## 4. Radial Fourier bands

`RadialFourierDecomposition` splits the coefficients into disjoint rings of radial frequency, measured in cycles per grid cell. Each band keeps the full layout, with coefficients outside it set to zero, so an inverse FFT shows what the band contributes in physical space. The bands sum back to the field when the final edge covers every mode, which in two dimensions needs an edge of at least $\sqrt{2}/2 \approx 0.71$.

In [ ]:
radial = RadialFourierDecomposition(edges=[0.08, 0.2, 0.75])
radial_components = radial(target, n_spatial_dims=2)
band_fields = {
    name: torch.fft.ifftn(band, dim=(2, 3), norm="ortho").real[0, -1, :, :, 0]
    for name, band in radial_components.items()
}
torch.testing.assert_close(sum(band_fields.values()), smoke_frame, atol=1e-5, rtol=1e-5)

band_titles = {
    "band_0": "band_0: below 0.08",
    "band_1": "band_1: 0.08 to 0.2",
    "band_2": "band_2: 0.2 and above",
}
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(
    smoke_frame,
    origin="lower",
    cmap="viridis",
    vmin=field_limits[0],
    vmax=field_limits[1],
)
axes[0].set_title("original smoke")
axes[1].imshow(
    band_fields["band_0"],
    origin="lower",
    cmap="viridis",
    vmin=field_limits[0],
    vmax=field_limits[1],
)
axes[1].set_title(band_titles["band_0"])
detail_limit = max(band_fields[name].abs().max() for name in ("band_1", "band_2"))
for axis, name in zip(axes[2:], ("band_1", "band_2"), strict=True):
    axis.imshow(
        band_fields[name],
        origin="lower",
        cmap="RdBu_r",
        vmin=-detail_limit,
        vmax=detail_limit,
    )
    axis.set_title(band_titles[name])
for axis in axes:
    axis.axis("off")
plt.tight_layout()

Bands can be scored like any other component. `CoefficientCRPSLoss` scores the ensemble distribution of each complex coefficient independently, and `TransformedLoss` averages the component scores.

In [ ]:
radial_crps = TransformedLoss(radial, CoefficientCRPSLoss())
print("Radial-band coefficient CRPS:", radial_crps(preds, target).item())

## 5. Discrete wavelets (DWT): localised scale components

Fourier coefficients describe the whole field at once. Wavelets are localised in both space and scale. A decimated discrete wavelet transform repeatedly splits each spatial axis into low-pass (`L`) and high-pass (`H`) responses. In two dimensions, the detail bands `LH`, `HL` and `HH` keep both scale and position. The final approximation is the coarsest low-pass field.

`WaveletDecomposition` uses differentiable `ptwt` transforms, supports one to three spatial axes, accepts named PyWavelets bases, and offers reflected or periodic padding. Components are named `level_<n>/<orientation>`, with level 1 the finest.

In [ ]:
dwt = WaveletDecomposition(wavelet="db2", levels=2, boundary="periodic")
dwt_components = dwt(target, n_spatial_dims=2)
for name, component in dwt_components.items():
    print(f"{name:24s} {tuple(component.shape)}")

In [ ]:
detail_names = list(dwt_components)[:3]
detail_bands = [dwt_components[name][0, -1, :, :, 0] for name in detail_names]
detail_limit = max(band.abs().max() for band in detail_bands)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for axis, name, band in zip(axes[:3], detail_names, detail_bands, strict=True):
    axis.imshow(
        band, origin="lower", cmap="RdBu_r", vmin=-detail_limit, vmax=detail_limit
    )
    axis.set_title(name)
    axis.axis("off")
approximation_name = "level_2/approximation"
axes[3].imshow(
    dwt_components[approximation_name][0, -1, :, :, 0],
    origin="lower",
    cmap="viridis",
)
axes[3].set_title(approximation_name)
axes[3].axis("off")
plt.tight_layout()

### Reconstruct each scale in physical space

Raw coefficients are filter responses on downsampled grids. To see what each scale contributes, keep one set of coefficients, set the rest to zero and apply the inverse transform. The reconstructed approximation and detail levels are back on the original grid, and they sum to the field. This view is for explanation only; the losses score the coefficients directly.

In [ ]:
dwt_coefficients = fswavedec2(
    smoke_frame, "db2", mode="periodic", level=2, axes=(-2, -1)
)


def reconstruct_dwt_component(
    *, approximation: bool = False, detail_level: int | None = None
) -> torch.Tensor:
    """Reconstruct the DWT approximation or one complete detail level."""
    isolated = [torch.zeros_like(dwt_coefficients[0])]
    isolated.extend(
        {name: torch.zeros_like(band) for name, band in details.items()}
        for details in dwt_coefficients[1:]
    )
    if approximation:
        isolated[0] = dwt_coefficients[0]
    if detail_level is not None:
        coefficient_index = len(dwt_coefficients) - detail_level
        isolated[coefficient_index] = dwt_coefficients[coefficient_index]
    return fswaverec2(tuple(isolated), "db2", axes=(-2, -1))


dwt_reconstructions = {
    "coarse approximation": reconstruct_dwt_component(approximation=True),
    "level 2 details": reconstruct_dwt_component(detail_level=2),
    "level 1 details": reconstruct_dwt_component(detail_level=1),
}
dwt_reconstructed = sum(
    dwt_reconstructions.values(), start=torch.zeros_like(smoke_frame)
)
torch.testing.assert_close(dwt_reconstructed, smoke_frame, atol=1e-5, rtol=1e-5)
print(
    "DWT reconstruction max error:",
    (dwt_reconstructed - smoke_frame).abs().max().item(),
)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(
    smoke_frame,
    origin="lower",
    cmap="viridis",
    vmin=field_limits[0],
    vmax=field_limits[1],
)
axes[0].set_title("original smoke")
axes[1].imshow(
    dwt_reconstructions["coarse approximation"],
    origin="lower",
    cmap="viridis",
    vmin=field_limits[0],
    vmax=field_limits[1],
)
axes[1].set_title("coarse approximation")
dwt_details = list(dwt_reconstructions.items())[1:]
dwt_detail_limit = max(detail.abs().max() for _, detail in dwt_details)
for axis, (name, detail) in zip(axes[2:], dwt_details, strict=True):
    axis.imshow(
        detail,
        origin="lower",
        cmap="RdBu_r",
        vmin=-dwt_detail_limit,
        vmax=dwt_detail_limit,
    )
    axis.set_title(name)
for axis in axes:
    axis.axis("off")
plt.tight_layout()

### Score the coefficients

A basic deterministic wavelet loss compares prediction and target coefficients using squared error. Here the first ensemble member supplies one forecast field, and `TransformedLoss` applies `CoefficientLpLoss(p=2)` to each DWT component with the default averaging.

In [ ]:
forecast = preds[..., 0].detach().requires_grad_()
dwt_loss = TransformedLoss(dwt, CoefficientLpLoss(p=2))
print("DWT coefficient squared error:", dwt_loss(forecast, target).item())

## 6. Dual-tree complex wavelets (DTCWT): adding direction

The dual-tree complex wavelet transform runs two real filter trees whose outputs combine into complex coefficients. At every level, the two-dimensional DTCWT gives six directional bands, oriented at roughly ±15°, ±45° and ±75°. A coefficient's magnitude describes how strongly a localised feature with that orientation is present. The transform is also nearly shift-invariant, unlike the decimated DWT.

`DTCWTDecomposition` uses `pytorch-wavelets`, accepts configurable first-level (`biort`) and deeper (`qshift`) filters, and can return the final low-pass. It works on real two-dimensional fields with symmetric boundaries. Components are named `level_<n>/direction_<k>` and `level_<levels>/lowpass`.

In [ ]:
dtcwt = DTCWTDecomposition(levels=3)
dtcwt_components = dtcwt(target, n_spatial_dims=2)
for name, component in dtcwt_components.items():
    print(f"{name:20s} {tuple(component.shape)} {component.dtype}")

In [ ]:
orientation_angles = (15, 45, 75, 105, 135, 165)
direction_magnitudes = [
    dtcwt_components[f"level_1/direction_{direction}"][0, -1, :, :, 0].abs()
    for direction in range(6)
]
magnitude_limit = max(magnitude.max() for magnitude in direction_magnitudes)
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for axis, angle, magnitude in zip(
    axes.flat, orientation_angles, direction_magnitudes, strict=True
):
    axis.imshow(magnitude, origin="lower", cmap="magma", vmin=0, vmax=magnitude_limit)
    axis.set_title(f"level 1: {angle}°")
    axis.axis("off")
plt.tight_layout()

### Reconstruct DTCWT scales in physical space

As with the DWT, the inverse DTCWT can reconstruct the low-pass or one complete directional level while the other coefficients are zero. These contributions are on the original grid and sum to the field. Reconstruction needs the complex coefficients; magnitudes alone are not enough.

In [ ]:
dtcwt_forward = DTCWTForward(
    J=3, biort="near_sym_a", qshift="qshift_a", mode="symmetric"
)
dtcwt_inverse = DTCWTInverse(biort="near_sym_a", qshift="qshift_a", mode="symmetric")
dtcwt_lowpass, dtcwt_highpasses = dtcwt_forward(smoke_frame[None, None])


def reconstruct_dtcwt_component(
    *, lowpass: bool = False, detail_level: int | None = None
) -> torch.Tensor:
    """Reconstruct the DTCWT low-pass or all directions at one level."""
    isolated_lowpass = dtcwt_lowpass if lowpass else torch.zeros_like(dtcwt_lowpass)
    isolated_highpasses = [torch.zeros_like(highpass) for highpass in dtcwt_highpasses]
    if detail_level is not None:
        isolated_highpasses[detail_level - 1] = dtcwt_highpasses[detail_level - 1]
    return dtcwt_inverse((isolated_lowpass, isolated_highpasses))[0, 0]


dtcwt_reconstructions = {
    "coarse low-pass": reconstruct_dtcwt_component(lowpass=True),
    "level 3 details": reconstruct_dtcwt_component(detail_level=3),
    "level 2 details": reconstruct_dtcwt_component(detail_level=2),
    "level 1 details": reconstruct_dtcwt_component(detail_level=1),
}
dtcwt_reconstructed = sum(
    dtcwt_reconstructions.values(), start=torch.zeros_like(smoke_frame)
)
torch.testing.assert_close(dtcwt_reconstructed, smoke_frame, atol=1e-5, rtol=1e-5)
print(
    "DTCWT reconstruction max error:",
    (dtcwt_reconstructed - smoke_frame).abs().max().item(),
)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(
    smoke_frame,
    origin="lower",
    cmap="viridis",
    vmin=field_limits[0],
    vmax=field_limits[1],
)
axes[0].set_title("original smoke")
axes[1].imshow(
    dtcwt_reconstructions["coarse low-pass"],
    origin="lower",
    cmap="viridis",
    vmin=field_limits[0],
    vmax=field_limits[1],
)
axes[1].set_title("coarse low-pass")
dtcwt_details = list(dtcwt_reconstructions.items())[1:]
dtcwt_detail_limit = max(detail.abs().max() for _, detail in dtcwt_details)
for axis, (name, detail) in zip(axes[2:], dtcwt_details, strict=True):
    axis.imshow(
        detail,
        origin="lower",
        cmap="RdBu_r",
        vmin=-dtcwt_detail_limit,
        vmax=dtcwt_detail_limit,
    )
    axis.set_title(name)
for axis in axes:
    axis.axis("off")
plt.tight_layout()

### Select components through the loss API

`ComponentBranch` selects named components, and `ComponentObjective` passes their score to `TransformedLoss`. This example selects the six directions at the finest level and scores them with coefficient-wise squared error. It is one standalone objective with default averaging.

In [ ]:
finest_components = [f"level_1/direction_{direction}" for direction in range(6)]
wavelet_loss = TransformedLoss(
    dtcwt,
    objective=ComponentObjective(
        [ComponentBranch(CoefficientLpLoss(p=2), components=finest_components)]
    ),
)

forecast.grad = None
wavelet_score = wavelet_loss(forecast, target)
wavelet_score.backward()
assert torch.isfinite(wavelet_score)
assert forecast.grad is not None
assert torch.isfinite(forecast.grad).all()
print("Finest-level coefficient squared error:", wavelet_score.item())
print("Gradient norm:                         ", forecast.grad.norm().item())

## 7. Physical-space scale bands

`ScaleDecomposition` takes smoothing operators from strongest to weakest and forms telescoping bands:

$$
z_1 = D_1 z, \qquad z_i = D_i z - D_{i-1} z, \qquad z_K = z - D_{K-1} z.
$$

The bands reconstruct the original field up to floating-point roundoff. Unlike spectral coefficients, they are physical-space fields, so ordinary spatial scores apply. The smoothing can be Gaussian or can keep the Fourier modes below a cutoff.

Gaussian `sigma` is in grid cells, and the Fourier `cutoff` is a radial frequency in cycles per grid cell. Both are relative to the grid, not fixed physical lengths. The Gaussian example uses reflected boundaries and the Fourier example is periodic.

In [ ]:
gaussian_scales = ScaleDecomposition(
    [
        ("large", GaussianSmoothing(sigma=4.0)),
        ("medium", GaussianSmoothing(sigma=1.5)),
    ],
    residual_name="small",
)
fourier_scales = ScaleDecomposition(
    [
        ("large", FourierLowPass(cutoff=0.08)),
        ("medium", FourierLowPass(cutoff=0.20)),
    ],
    residual_name="small",
)

for label, decomposition in {
    "Gaussian": gaussian_scales,
    "Fourier": fourier_scales,
}.items():
    components = decomposition(target, n_spatial_dims=2)
    reconstruction = sum(components.values(), start=torch.zeros_like(target))
    torch.testing.assert_close(reconstruction, target, atol=1e-6, rtol=1e-6)
    error = (reconstruction - target).abs().max()
    print(f"{label:8s} bands={tuple(components)} max error={error.item():.3e}")

In [ ]:
scale_images = {
    name: component[0, -1, :, :, 0]
    for name, component in gaussian_scales(target, n_spatial_dims=2).items()
}
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(scale_images["large"], origin="lower", cmap="viridis")
axes[0].set_title("Gaussian large")
scale_limit = max(scale_images[name].abs().max() for name in ("medium", "small"))
for axis, name in zip(axes[1:], ("medium", "small"), strict=True):
    axis.imshow(
        scale_images[name],
        origin="lower",
        cmap="RdBu_r",
        vmin=-scale_limit,
        vmax=scale_limit,
    )
    axis.set_title(f"Gaussian {name}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

scale_crps = TransformedLoss(gaussian_scales, CRPSLoss())
print("Gaussian-band CRPS:", scale_crps(preds, target).item())

## 8. Effect of smoothing on the decomposition

Smoothing removes fine-scale variability from a field. Below, each ensemble member is smoothed with a Gaussian of width 1.5 grid cells. The field plots and per-scale diagnostics show where that variability is lost.

In [ ]:
smoothed_preds = GaussianSmoothing(sigma=1.5)(preds.detach(), n_spatial_dims=2)

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
images = [
    (smoke_frame, "Target smoke"),
    (preds[0, -1, :, :, 0, 0].detach(), "Sharp member"),
    (smoothed_preds[0, -1, :, :, 0, 0], "Smoothed member"),
]
for axis, (image, title) in zip(axes, images, strict=True):
    axis.imshow(
        image,
        origin="lower",
        cmap="viridis",
        vmin=field_limits[0],
        vmax=field_limits[1],
    )
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()

Two diagnostics of variability at each scale: the radially averaged Fourier power spectrum and the root-mean-square DTCWT magnitude at each level. The sharp and smoothed ensembles can be compared with the target. These are visual summaries of the decompositions, rather than additional loss terms.

In [ ]:
levels = (1, 2, 3)


def as_fields(fields: torch.Tensor) -> torch.Tensor:
    """Fold any ensemble axis into the batch, giving (B, T, H, W, C)."""
    fields = fields.detach()
    if fields.ndim == 6:
        fields = rearrange(fields, "b t h w c m -> (b m) t h w c")
    return fields


def radial_power_spectrum(fields: torch.Tensor) -> torch.Tensor:
    """Mean Fourier power in integer rings of radial wavenumber."""
    fields = as_fields(fields)
    height, width = fields.shape[2:4]
    power = torch.fft.fft2(fields, dim=(2, 3), norm="ortho").abs().square()
    power = power.mean(dim=(0, 1, 4))
    ky = torch.fft.fftfreq(height) * height
    kx = torch.fft.fftfreq(width) * width
    rings = (ky[:, None].square() + kx[None, :].square()).sqrt().round().long()
    totals = torch.zeros(int(rings.max()) + 1).index_add_(
        0, rings.flatten(), power.flatten()
    )
    counts = torch.bincount(rings.flatten()).clamp_min(1)
    return totals / counts


def level_rms(fields: torch.Tensor) -> list[float]:
    """Root-mean-square DTCWT magnitude at each level, over all directions."""
    components = dtcwt(as_fields(fields), n_spatial_dims=2)
    return [
        torch.stack(
            [
                components[f"level_{level}/direction_{direction}"].abs().square().mean()
                for direction in range(6)
            ]
        )
        .mean()
        .sqrt()
        .item()
        for level in levels
    ]


nyquist = smoke_frame.shape[0] // 2
wavenumbers = torch.arange(1, nyquist + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for label, fields, style in (
    ("target", target, "k-"),
    ("sharp ensemble", preds, "C0-"),
    ("smoothed ensemble", smoothed_preds, "C3-"),
):
    spectrum = radial_power_spectrum(fields)[1 : nyquist + 1]
    axes[0].loglog(wavenumbers, spectrum, style, label=label)
    axes[1].semilogy(levels, level_rms(fields), style + "o", label=label)
axes[0].set_xlabel("radial wavenumber")
axes[0].set_ylabel("mean Fourier power")
axes[0].set_title("Fourier power spectrum")
axes[1].set_xticks(levels, ["1 (finest)", "2", "3 (coarsest)"])
axes[1].set_xlabel("DTCWT level")
axes[1].set_ylabel("RMS coefficient magnitude")
axes[1].set_title("DTCWT magnitude by level")
axes[0].legend()
plt.tight_layout()

## Notes and limits

- Fourier decompositions treat the field as periodic. Fields with walls or open boundaries pick up some energy from the seam.
- Radial bands keep every coefficient position, zeroed outside the band, so band means include those zeros; `band_mean=True` rescales coefficients by band size.
- `WaveletDecomposition` supports one to three spatial axes with reflected or periodic padding. `DTCWTDecomposition` takes real two-dimensional fields with symmetric boundaries.
- The wavelet backends support only single and double precision, so half-precision inputs are promoted to float32 and run outside reduced-precision autocast.
- Component selectors are explicit names, not patterns.
- Means over coefficients and grid cells weight every cell equally; they are not area-weighted.

See the reference pages on [spectral losses](../docs/reference/losses/spectral.md), [scale losses](../docs/reference/losses/scale.rst) and [wavelet losses](../docs/reference/losses/wavelet.md) for the full interfaces.